<a href="https://colab.research.google.com/github/Ademola-Olorunnisola/Nonlinear_and_Data_Driven_Estimation/blob/main/Ademola_Nonlinear_Observability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Symbolic nonlinear observability

In [34]:
import numpy as np
import sympy as sp
from IPython.display import display

In [35]:
# Import functions directly from github
# Important: note that we use raw.githubusercontent.com, not github.com

import requests
url = 'https://raw.githubusercontent.com/florisvb/Nonlinear_and_Data_Driven_Estimation/main/Utility/symbolic_derivatives.py'
r = requests.get(url)

# Store the file to the colab working directory
with open('symbolic_derivatives.py', 'w') as f:
    f.write(r.text)

# import the function we want from that file
import symbolic_derivatives

# Example: downward facing constant altitude monocular camera

Here we have a single camera pointed down, moving laterally at constant altitude.

$
\mathbf{\dot{x}} = \mathbf{f}(\mathbf{x},\mathbf{u}) =
\frac{d}{dt}
\begin{bmatrix}
\bbox[yellow]{g} \\[0.3em]
\bbox[yellow]{z} \\[0.3em]
\end{bmatrix} =
\overset{f_0}{\begin{bmatrix}
0 \\[0.3em]
0
\end{bmatrix}} +
\overset{f_1}{\begin{bmatrix}
1 \\[0.3em]
0
\end{bmatrix}} \bbox[lightgreen]{u}
$

We have 1 measurement, the ventral optic flow. We can also assume we have lateral acceleration measurements, but in this case the acceleration is entirely defined by the control inputs we don't have to add it explicitly.

$
\mathbf{y} = \mathbf{{h}}(\mathbf{{x}}, \mathbf{{u}}) =
\begin{bmatrix}
\bbox[yellow]{g/z} \\[0.3em]
\end{bmatrix}
$

# Define states and dynamics in control affine form

In [36]:
# Define states
S, V, I, R, gamma, sigma = sp.symbols(['S', 'V', 'I', 'R', 'gamma', 'sigma'])
x = [S, V, I, R, gamma, sigma]

# Parameters
Lambda, mu, beta = sp.symbols(['Lambda', 'mu', 'beta'])
N = sp.symbols('N')  # Total population

f_0 = sp.Matrix([Lambda - beta * S * I - mu * S,
                  -sigma * V * I - mu * V,
                  beta*S*I + sigma*V*I - gamma*I - mu*I,
                  gamma*I - mu*R,
                  0,
                  0])

f_1 = sp.Matrix([
    -S,      # this gets multiplied by α
    S,
    0,
    0,
    0,
    0
])

f_2 = sp.Matrix([
    beta*S*I,                       # this gets multiplied by κ
    sigma*V*I,
    -beta*S*I - sigma*V*I,
    0,
    0,
    0
])

In [37]:
x0 = {
    S: 178400000,
    V: 22300000,
    I: 40006200,
    R: 10704000,
    beta: 0.05,
    sigma: 0.0001375,
    Lambda: 9.04e-5,
    mu: 4.3e-5,
    gamma: 0.00555,
    N: 223000000
}

# Define measurements

In [38]:
# Measurement function
N = sp.symbols('N')  # Total population (constant)

h = sp.Matrix([I, V, R])   # y₂: Vaccination coverage ratio

# Calculate each term in G

$G = [h, L_{f_0}h, L_{f1}h]$

In [39]:
# Calculate each term in G
# G = [h, L_{f_0}h, L_{f_1}h, L_{f_2}h]

# Take the derivative of h with respect to x along the vector f_0
L_f0_h = symbolic_derivatives.directional_derivative(h, x, f_0)
display(L_f0_h)

print('')

# Take the derivative of h with respect to x along the vector f_1
L_f1_h = symbolic_derivatives.directional_derivative(h, x, f_1)
display(L_f1_h)

print('')

# Take the derivative of h with respect to x along the vector f_2
L_f2_h = symbolic_derivatives.directional_derivative(h, x, f_2)
display(L_f2_h)

Matrix([
[I*S*beta + I*V*sigma - I*gamma - I*mu],
[                    -I*V*sigma - V*mu],
[                       I*gamma - R*mu]])

Matrix([
[0],
[S],
[0]])

Matrix([
[-I*S*beta - I*V*sigma],
[            I*V*sigma],
[                    0]])

# Assemble G, take Jacobian

In [40]:
# Assemble G, take Jacobian
G = sp.Matrix([h, L_f0_h])
display(G)

Matrix([
[                                    I],
[                                    V],
[                                    R],
[I*S*beta + I*V*sigma - I*gamma - I*mu],
[                    -I*V*sigma - V*mu],
[                       I*gamma - R*mu]])

In [41]:
# Jacobian of G with respect to states
display(G.jacobian(x))

Matrix([
[     0,             0,                             1,   0,  0,    0],
[     0,             1,                             0,   0,  0,    0],
[     0,             0,                             0,   1,  0,    0],
[I*beta,       I*sigma, S*beta + V*sigma - gamma - mu,   0, -I,  I*V],
[     0, -I*sigma - mu,                      -V*sigma,   0,  0, -I*V],
[     0,             0,                         gamma, -mu,  I,    0]])

# Check the rank of G for a given operating point, $x_0$

In [42]:
# Check the rank of G for a given operating point, x_0
# Using realistic values from Nigeria TB data

x0 = {
    S: 178400000,
    V: 22300000,
    I: 40006200,
    R: 10704000,
    beta: 0.05,
    sigma: 0.0001375,
    Lambda: 9.04e-5,
    mu: 4.3e-5,
    gamma: 0.00555,
    N: 223000000
}

display(G.jacobian(x).subs(x0))

print('')
print('Rank of G:')
G.jacobian(x).subs(x0).rank()

Matrix([
[        0,            0,              1,       0,         0,                0],
[        0,            1,              0,       0,         0,                0],
[        0,            0,              0,       1,         0,                0],
[2000310.0,    5500.8525, 8923066.244407,       0, -40006200,  892138260000000],
[        0, -5500.852543,       -3066.25,       0,         0, -892138260000000],
[        0,            0,        0.00555, -4.3e-5,  40006200,                0]])


Rank of G:


6

# Shortcut function to get G:

### First derivatives

In [43]:
# First derivatives
G1 = symbolic_derivatives.get_bigO(h, x, [f_0, f_1, f_2])

# Second derivatives
G2 = symbolic_derivatives.get_bigO(sp.Matrix.vstack(*G1), x, [f_0, f_1, f_2])

# Both first and second derivatives
G = sp.Matrix.vstack(*G1, *G2)
display(G)

Matrix([
[                                                                                                                                         I],
[                                                                                                                                         V],
[                                                                                                                                         R],
[                                                                                                     I*S*beta + I*V*sigma - I*gamma - I*mu],
[                                                                                                                         -I*V*sigma - V*mu],
[                                                                                                                            I*gamma - R*mu],
[                                                                                                                                         0

# Exercises:

1. Is the system observable with no controls (i.e. $u=0$)?
2. Is the system observable with control? (i.e. $u\neq0$)?
3. How many derivatives are needed, 1 or 2?
3. Apply the symbolic approach to the planar drone example with the measurements below. What is necessary in order for $z$ to be observable?